# Quantity as a function of other Quantities — design exploration

*Working notebook for the question in Mike's notes (2026-05-31): "a quantity should be a function of another quantity or combination of quantities — how do we enable that?"*

The sketch in the notes was:

```python
Vout: Quantity = Quantity(
    descriptor = "voltage divider output",
    unit       = units.V,
    value      = [a function maybe!]   # <-- the open question
    by_mode    = {...},
)
```

The key realization this notebook works through: **"a Quantity that is a function of other Quantities" is three different ideas, and the framework already does two of them.** Only the third (literally storing a function inside `value`) is new — and it fights the architecture in specific, demonstrable ways.

| Idea | What it means | Status |
|---|---|---|
| **1. Eager arithmetic** | `a + b` returns a *computed* Quantity | ✅ exists today |
| **2. The function is a DAG node** | `def vout(vin, r1, r2): ...` wired by Hamilton | ✅ exists today (the intended model) |
| **3. `value` holds a callable** | the Quantity is *lazy/symbolic*, recomputes itself | ❓ what the note sketches — examined below |

In [1]:
from framework.quantity import Constant, Quantity, RangeQuantity, INVARIANT
from framework import units
from framework.units import V, Ohm, kOhm, K, registry

# Canonical import path is `from framework...` (the installed editable package).
# Do NOT also import via `hw_analysis_framework.src.framework` — that creates a
# second Pint registry and cross-registry comparisons fail silently.

## 1. Eager arithmetic already makes a Quantity a function of Quantities

`a + b`, `a * b`, `a / b` between Quantities return a **new, fully-evaluated Quantity** — units propagated, scenario/mode axes merged. So the voltage divider already works, today, with no new machinery:

In [2]:
vin = Quantity(unit=V, by_scenario={'cold': 9.0, 'hot': 16.0})   # supply varies by corner
r1  = Constant(10, kOhm)
r2  = Constant(10, kOhm)

vout = vin * r2 / (r1 + r2)        # <-- 'a function of other quantities', evaluated eagerly
print('vout      :', vout)
print('unit      :', vout.unit)
print('at hot    :', vout.at(scenario='hot'))

vout      : {cold: 4.5, hot: 8.0} V
unit      : V
at hot    : 8.0


Notice what `vout` *is*: a Quantity whose `value`/`by_scenario` holds the **computed numbers** `{cold: 4.5, hot: 8.0}`. 

So the answer to *"is the `value` parameter just `a + b` where a and b are Quantities?"* is: **`value` holds the *result* of `a + b`, not the unevaluated expression.** The expression ran the moment you wrote it.

In [3]:
# value is the computed payload, not a formula:
print('vout.by_scenario :', vout.by_scenario)
print('vout.value       :', vout.value)   # None here because the value lives on the scenario axis

vout.by_scenario : {'cold': 4.5, 'hot': 8.0}
vout.value       : None


## 2. Where the *function* actually lives: the DAG node

If you want to **name** the relationship and reuse it — "Vout is a function of Vin, R1, R2" — you write a plain Python function. That function *is* the thing your note calls `[a function maybe!]`, but it lives **one level up** from the Quantity, as a DAG node:

In [4]:
def vout(vin: Quantity, r1: Quantity, r2: Quantity) -> Quantity:
    """Voltage-divider output. A normal function — returns an eager Quantity."""
    return vin * r2 / (r1 + r2)

# Called directly here. In a block, Hamilton wires `vin`, `r1`, `r2` BY PARAMETER NAME
# to other nodes of the same name — no registration, just literal name matching.
print(vout(vin, r1, r2))

{cold: 4.5, hot: 8.0} V


This is the answer to *"do we just define a function and calculate the quantity?"* — **yes, and that is the framework's intended model.** The function is a node in the Hamilton DAG (design doc §7). Benefits you get for free by keeping it here rather than inside the Quantity:

- **Provenance** links each result Quantity back to its producing node (design doc §9).
- **Caching** is content-addressed on the *function's source bytes* + ancestor hashes (design doc §8). The function is the unit of caching.
- **Axis propagation** happens automatically — change `vin` to vary by mode and `vout` follows, no edits:

In [5]:
vin_by_mode = Quantity(unit=V, by_mode={
    'crank': Quantity(unit=V, by_scenario={'cold': 6.0, 'hot': 7.5}),   # brown-out during crank
    'run':   Quantity(unit=V, by_scenario={'cold': 13.0, 'hot': 16.0}),
})
print(vout(vin_by_mode, r1, r2))   # same function; mode + scenario axes both propagate

{crank: {cold: 3.0, hot: 3.75}, run: {cold: 6.5, hot: 8.0}} V


## 3. The notes' harder example: Vout depends on Vin **and the temperature of the resistors**

From the notes: *"what would this look like for a voltage that depends on an input voltage and the temperature of a resistor(s)?"* — with R1/R2 imported via the BOM and their resistance computed at the project level from ambient temperature.

This also already works, because temperature variation is just another Quantity flowing through the same eager arithmetic. The resistor's *effective* resistance is itself a function of a Quantity (ΔT), and the scenario axis carries through to `vout` untouched:

In [6]:
# Tempco model: R_eff = R_nom * (1 + tempco * ΔT).
# (We use ΔT in kelvin, not absolute degC, to dodge Pint's offset-unit math — see framework CLAUDE.md gotcha #2.)
per_K  = registry.parse_units('1/K')
tempco = Constant(100e-6, per_K)                                  # 100 ppm/K
dT     = Quantity(unit=K, by_scenario={'cold': -40.0, 'hot': 60.0})  # ambient relative to 25 C ref

def r_eff(r_nom: Quantity, tempco: Quantity, dT: Quantity) -> Quantity:
    return r_nom * (1 + tempco * dT)

r1_eff = r_eff(Constant(10, kOhm), tempco, dT)
r2_eff = r_eff(Constant(10, kOhm), tempco, dT)
print('R1 effective :', r1_eff)
print('Vout         :', vout(vin, r1_eff, r2_eff))   # scenario axis flows all the way through

R1 effective : {cold: 9.96, hot: 10.06} kΩ
Vout         : {cold: 4.5, hot: 8.0} V


So the **physics of the notes' example needs nothing new.** Temperature is a Quantity; resistance-of-temperature is a function returning an eager Quantity; the divider consumes it. The composition is the DAG.

> The genuinely *open* part of that note — *"blocks can have their own modes ... determined by inputs from other blocks"* — is **derived modes**, a separate feature (see §6). Today `by_mode` is a static `name → Quantity` dict; *computing which mode you're in from inputs* is not modeled.

## 3a. Following the thread: make the *input* itself a function — `Vin = f(temperature, ignition state)`

Back in §1, `vin` was a hand-written leaf: `Quantity(by_scenario={'cold': 9.0, 'hot': 16.0})`. But those numbers aren't independent facts — the battery rail *depends on* project-level **temperature** and on a discrete **ignition state** (key on → alternator charging; key off → battery only). So turn the leaf into a node, and let both axes carry it:

- **temperature → `by_scenario` (inner) + a project input.** `ambient_temp` is exactly the key the framework auto-supplies to every block (`Project.standard_inputs()` pulls the scenario keys present on *every* scenario — see framework CLAUDE.md gotcha #5). You don't pass it around; you name the parameter `ambient_temp` and Hamilton wires it. The cold/hot corners live in `scenarios.toml`.
- **key on / key off → `by_mode` (outer).** A discrete operating state — exactly the `IGNITION_VOLTAGE` case from the notes.

They **nest**: mode outer, scenario inner. Same shape the framework already supports.

In [7]:
from framework.analyses import voltage_divider
per_K = registry.parse_units('1/K')

# Temperature is a PROJECT-LEVEL scenario input (cold/hot corners come from scenarios.toml).
ambient_temp = Quantity(unit=K, by_scenario={'cold': 233.15, 'hot': 358.15})   # -40 C / +85 C

def battery_rail(ambient_temp: Quantity) -> Quantity:
    """Vin as a function of temperature AND ignition state. Belongs at PROJECT level
    (or published as a Contract by the rail owner) and consumed by blocks."""
    t_ref = Constant(298.15, K)                  # 25 C reference
    tc    = Constant(-0.0008, per_K)             # illustrative: rail sags when cold
    frac  = 1 + (ambient_temp - t_ref) * tc      # dimensionless, varies by scenario
    return Quantity(unit=V, by_mode={
        'key_on':  Constant(14.4, V) * frac,     # alternator charging
        'key_off': Constant(12.6, V) * frac,     # battery only
    })

vin_node = battery_rail(ambient_temp)            # replaces the hand-written §1 leaf
vout_node = voltage_divider(vin_node, Constant(10, kOhm), Constant(10, kOhm))
print('vin  :', vin_node)
print('vout :', vout_node)
print('vin @ (key_off, cold):', vin_node.at(scenario='cold', mode='key_off'))

vin  : {key_on: {cold: 15.148800000000001, hot: 13.7088}, key_off: {cold: 13.2552, hot: 11.995199999999999}} V
vout : {key_on: {cold: 7.5744, hot: 6.8544}, key_off: {cold: 6.627599999999999, hot: 5.997599999999999}} V
vin @ (key_off, cold): 13.2552


The scenario axis (temperature) flows through `frac` into `vin` and onward into `vout` untouched, while the mode axis (ignition) rides on top. That's the holistic model the notes describe — `vin = f(temperature, ignition_state)` — with the *values computed from a project input* rather than typed in by hand.

| Concept | Axis | Why |
|---|---|---|
| temperature | `by_scenario` (inner) + project input | "what my surroundings are doing"; continuous; shared across all blocks |
| key on / key off | `by_mode` (outer) | "what I'm doing"; discrete operating state |

**The real decision lurking here — mode vs. scenario for the discrete state:**

- **key on/off as a *mode*** (above) nests cleanly and matches the notes' taxonomy. Recommended.
- **key on/off as a *named scenario*** crossed with temperature puts you in a **product space** — `{cold·keyon, cold·keyoff, hot·keyon, hot·keyoff}`. Today `by_scenario` is a flat string dict, so you'd hand-enumerate compound keys. That is exactly the **page-1 open question** (*"overlapping / inter-related scenarios... 'hot' AND 'reverse battery'... maybe scenarios should be encoded in variables"*). Mode-vs-scenario is the framework's current answer — good for *one* discrete state crossed with the corners, but two-plus interacting environmental states (key-state × reverse-battery × load-dump) is genuinely unmodeled. See §6.4.

> Placement: per the notes (*"Vin is imported as a Quantity"*), `battery_rail` lives at **project level** or is **published as a Contract** by whoever owns the rail, and is *consumed* by the divider block — keeping one owner per physical rail (the block-ownership rule).

## 3b. Why this kills the "mess of named scenarios"

The hang-up worth naming explicitly. A leaf like `v_bat = Quantity(by_scenario={'cold': 9.0, 'hot': 16.0})` is the **anti-pattern**: the corners are hand-typed *on the leaf*, decoupled both from each other and from the temperature that actually drives them. Do that on every leaf and a large project really does become a swamp of magic numbers.

The fix is not to remove `by_scenario` — it's to author the corners in **one** place. `ambient_temp` is itself a `by_scenario` Quantity, *machine-generated* from `scenarios.toml`. Make every temperature-dependent value a **function** of it (§3a), and the named corners collapse to a single root axis; everything downstream is fluid math. Watch the messy leaf turn out to be nothing but `battery_rail(ambient_temp)` evaluated by hand:

In [8]:
# ❌ MESS: corners hand-typed on a leaf — magic numbers, decoupled from their cause
v_bat_messy = Quantity(unit=V, by_scenario={'cold': 15.1488, 'hot': 13.7088})

# ✅ FLUID: one root axis (ambient_temp, from scenarios.toml); the value is computed
v_bat_fluid = battery_rail(ambient_temp).by_mode['key_on']   # reuse the §3a function

print('hand-typed leaf :', v_bat_messy)
print('computed value  :', v_bat_fluid)   # identical — the magic numbers WERE this formula

hand-typed leaf : {cold: 15.1488, hot: 13.7088} V
computed value  : {cold: 15.148800000000001, hot: 13.7088} V


**The count of named scenarios equals the number of distinct environmental worldviews you choose to analyze — it does *not* grow with the number of derived quantities.** The explosion you might fear comes from naming *combinations* (`hot_reverse_battery_highload`). The discipline that prevents it:

1. Keep scenarios as a **few orthogonal variables** (`ambient_temp`, `vbat_source`, `load`), defined once in `scenarios.toml`.
2. Author each leaf as a **function of whichever variables actually affect it**.
3. Reserve hand-typed `by_scenario` for irreducible curated bundles where no cleaner function exists.

> **The one genuine friction:** *smooth math* propagates through `by_scenario` perfectly, but a real branch — `if ambient_temp > 0: ...` — does **not**, because `ambient_temp` carries every corner at once in a single DAG run, not a scalar you can test. Express piecewise behavior as propagating math (`min`/`max`/clamp) where you can; genuine conditionals are the strongest case for a per-scenario execution loop (see §6).

## 3c. Does the worldview stay consistent down the chain? (correlation + missing keys)

Two correctness questions fall out of all this — and both are answered by `_combine_scenarios` / `_aligned_scenario_keys`.

**(1) Correlation.** When two `by_scenario` Quantities combine, they join on the scenario **name**. `cold` only ever meets `cold`; there is no code path that pairs `cold` with `hot`. Because every intermediate result is itself keyed by the same names, this holds **transitively** down an arbitrarily long chain — the voltage can never pick the cold corner while the current picks hot.

In [9]:
from framework.units import mA

volt = Quantity(unit=V,  by_scenario={'cold': 9.0,  'hot': 16.0})
curr = Quantity(unit=mA, by_scenario={'cold': 5.0,  'hot': 20.0})
print('correlated product :', volt * curr)   # {cold: 9*5, hot: 16*20} — never 9*20 or 16*5

correlated product : {cold: 45.0, hot: 320.0} mA * V


**(2) Missing keys** — what happens when an operand has nothing for a given scenario. Four distinct behaviors:

In [10]:
def show(label, fn):
    try: print(f'{label:42s} -> {fn()}')
    except Exception as e: print(f'{label:42s} -> {type(e).__name__}: {e}')

# A) a plain Constant broadcasts to every scenario
show('Constant broadcast', lambda: volt * Constant(2.0, mA))
# B) explicit INVARIANT (_ALL_) is the fallback for unnamed scenarios
show('INVARIANT fallback', lambda: volt * Quantity(unit=mA, by_scenario={'cold': 5.0, INVARIANT: 99.0}))
# C) disjoint / partial overlap is refused with a clear error
show('disjoint sets',      lambda: volt * Quantity(unit=mA, by_scenario={'warm': 7.0}))
# D) strict subset, no _ALL_ -> clear ValueError (fixed 2026-05-31; was a cryptic KeyError)
show('subset, no _ALL_',   lambda: volt * Quantity(unit=mA, by_scenario={'cold': 5.0}))

Constant broadcast                         -> {cold: 18.0, hot: 32.0} mA * V
INVARIANT fallback                         -> {cold: 45.0, hot: 1584.0} mA * V
disjoint sets                              -> ValueError: Cannot combine Quantities with disjoint scenario sets: {'hot', 'cold'} vs {'warm'}
subset, no _ALL_                           -> ValueError: Cannot combine Quantities: one operand has no value for scenario(s) ['hot'] and no scenario-invariant fallback. Name the missing scenario(s), use a Constant, or add an INVARIANT entry.


| Operand shape | Behavior |
|---|---|
| `Constant` / no axis | **broadcasts** to every scenario |
| has explicit `INVARIANT` (`_ALL_`) | named keys win; `_ALL_` is the **fallback** for the rest |
| disjoint or partially-overlapping keys | **`ValueError`** — refuses to silently guess ✅ |
| strict subset of the other's keys, **no** `_ALL_` | **`ValueError`** — names the undefined scenario(s) (fixed 2026-05-31) ✅ |

So to say "this value is scenario-independent," use a `Constant` (broadcasts) or an explicit `INVARIANT` entry (fallback). The last row *used* to be a rough edge: `_aligned_scenario_keys` permitted a subset assuming an `INVARIANT` entry existed, but nothing enforced it — so `{cold}` combined with `{cold, hot}` died in `_pick` with a bare `KeyError`. **Fixed 2026-05-31:** alignment now rejects a subset without an `INVARIANT` fallback up front, naming the undefined scenario(s).

## 4. The tempting third option: a function *inside* `value` (lazy / symbolic Quantity)

This is what the note literally sketches — `value = [a function maybe!]`. It would make a Quantity **lazy**: it stores a closure and recomputes itself on demand. It's seductive (the Quantity becomes self-describing), but it collides with three load-bearing properties of the current design. Let's make each collision concrete.

**Collision 1 — caching & immutability.** `Quantity` is a `frozen` dataclass and the cache is content-addressed: it hashes/pickles Quantities by *value*. A closure is neither stably hashable nor picklable, so a Quantity carrying one can't be cached or serialized:

In [11]:
import pickle
try:
    pickle.dumps(lambda: vin * r2)        # a 'value = function' payload
    print('lambda pickled (unexpected)')
except Exception as e:
    print('closure pickle FAILS:', type(e).__name__, '-> uncacheable, unserializable')

print('plain Constant pickles fine     :', bool(pickle.dumps(Constant(5, V))))

closure pickle FAILS: PicklingError -> uncacheable, unserializable
plain Constant pickles fine     : True


**Collision 2 — construction-time guarantees.** Today `Quantity.__post_init__` validates units and axis structure *the moment you build it*. A lazy `value` defers all of that to call time — you lose the "if it constructed, it's well-formed" contract.

**Collision 3 — provenance is the DAG's job.** Provenance currently links a Quantity to its **producing DAG node**. A self-recomputing Quantity would have to introspect its own closure to rebuild lineage — i.e. **reimplement Hamilton inside `Quantity`.** And note the existing cost of the *eager* model, visible right now:

In [12]:
print('DAG-node result provenance :', '(attached by Project.run)')
print('ad-hoc arithmetic result   :', vout(vin, r1, r2).provenance)
# ^ '<literal>': Quantities built by bare +-*/ outside Project.run carry no lineage
#   (framework CLAUDE.md gotcha #10). The DAG is what supplies provenance — which is
#   exactly the role a 'value = function' Quantity would be trying to duplicate.

DAG-node result provenance : (attached by Project.run)
ad-hoc arithmetic result   : ProvenanceRef(node_id='<literal>', label=None)


## 5. The middle ground that stays eager: reusable formula functions

There's a real ergonomic want underneath the note — *author a formula once, reuse it* — and it's already served by the **standard-analyses library** (design doc §11). These are ordinary functions that take Quantities and return eager Quantities. Author once, call anywhere; drop into a block and it becomes a DAG node automatically:

In [13]:
from framework.analyses import voltage_divider

print(voltage_divider(vin, r1_eff, r2_eff))   # the §3 result, via the library helper
print(voltage_divider.__doc__.strip().splitlines()[0])

{cold: 4.5, hot: 8.0} V
Resistive voltage divider: ``V_out = V_in * R_bottom / (R_top + R_bottom)``.


This is *"define a function and calculate the quantity"* done in the grain of the framework: the function is reusable and testable, it returns eager Quantities, and it carries provenance/caching when it runs as a node. No new field on `Quantity` required.

## 6. What's actually still open (decisions for Mike, not code)

The "function of Quantities" question resolves to *use eager arithmetic + DAG nodes*. But the notes surface three genuinely-unmodeled threads worth their own design passes:

1. **Derived / computed modes.** *"blocks can have their own modes ... determined by inputs from other blocks or the project."* Today `by_mode` is a static `name → Quantity` dict. Selecting/deriving a mode *from* input Quantities (e.g. an `IGNITION_VOLTAGE`-driven state) is a new capability — closer to a small state function than to the `Quantity` type.

2. **Distributions (the page-1 characterization idea).** Design doc §6.1 *already reserves* a `distribution: Distribution | None` field on `Quantity` (deferred in the current build). The notes' framing — **"characterization tells us how it really behaves (samples across lots), with a distribution; the synthetic spec/datasheet analysis is the *contract*"** — maps directly onto that field: arithmetic would *convolve* distributions (design doc §7.5 Monte Carlo, opt-in). This is the natural home for the characterization-vs-contract duality.

3. **Provenance through arithmetic.** If we ever want ad-hoc derived Quantities (not just DAG nodes) to carry lineage, we'd thread provenance through `_binop` (gotcha #10). Decision: is DAG-level provenance enough, or do we want it on every `+ - * /`?

4. **Scenario modeling (page-1 open question).** *"how to handle overlapping / inter-related scenarios? i.e. 'hot' AND 'reverse battery' ... maybe scenarios should never be named but encoded in variables."* This is a real fork in how worst-case corners are represented — named corners vs. a variable-space the corners are *drawn from*. Bigger than Quantity; flagged here because the note raises it.

## Summary

- **You already have "a Quantity that is a function of other Quantities."** Eager arithmetic (`a*b`) computes it; a DAG node (`def vout(...)`) names and reuses it. That is the intended model.
- **`value` holds the *result*, not the formula.** Putting a *function* in `value` makes the Quantity lazy/symbolic and breaks caching, construction-time validation, and provenance — it reinvents the DAG inside the type. Recommend **not** doing this.
- **The reusable-formula want is met by the standard-analyses library** (eager functions in, eager Quantities out).
- **The real open work** is derived modes, distributions (characterization vs. contract — the field is already reserved in §6.1), and scenario modeling. Those deserve their own design passes.